###Question 1 - Principles of Data Science CS5530
Morgan Sansone

In [ ]:
import pandas as pd
import numpy as np
import io

##Ingest

In [ ]:
# raw data
raw_data = {
    'Height': [65.8, 71.5, 69.4, 68.2, 67.8, 68.7, 69.8, 70.1, 67.9, 66.8],
    'Weight': [112, 136, 153, 142, 144, 123, 141, 136, 112, 120],
    'Age': [30, 19, 45, 22, 29, 50, 51, 23, 17, 39],
    'Grip strength': [30, 31, 29, 28, 24, 26, 22, 20, 19, 31],
    'Frailty': ['N', 'N', 'N', 'Y', 'Y', 'N', 'Y', 'Y', 'N', 'N']
}

# create dataframe loading the raw_data into
df = pd.DataFrame(raw_data)

# save to csv file frailty_data.csv
df.to_csv('frailty_data.csv', index=False)

# read into dataframe
df = pd.read_csv('frailty_data.csv')
df.shape # show the shape rows, columns

(10, 5)

##Process

In [ ]:
# creating a deep copy so the original dataframe preserved
df_copy = df.copy()

# unit standardization
df_copy['Height_m'] = df_copy['Height'] * 0.0254
df_copy['Weight_kg'] = df_copy['Weight'] * 0.45359237

# feature engineering BMI & age grouping

# BMI calculations rounded to 2 decimals
df_copy['BMI'] = (df_copy['Weight_kg'] / (df_copy['Height_m'] ** 2)).round(2) # dataframe

# age grouping
bins = [0, 29, 45, 60, 120] # creating the ranges 0-29, 30-45, 46-60, 60 and up
labels = ['<30', '30–45', '46–60', '>60'] # create the labels
df_copy['AgeGroup'] = pd.cut(df_copy['Age'], bins=bins, labels=labels) # create the dataframe

# categorical to numeric encoding
df_copy['Frailty_binary'] = df_copy['Frailty'].map({'Y': 1, 'N': 0}).astype('int8')

# one-hot encode age group into columns
df_copy = pd.get_dummies(df_copy, columns=['AgeGroup'], prefix='AgeGroup')

# drop old columns to only have new columns made (height,weight,age,fraility)
df_copy.drop(['Height', 'Weight', 'Age', 'Frailty'], axis=1, inplace=True)

# save to table/cleaned version
df_copy.to_csv('frailty_data_cleaned.csv', index=False)

##Analyze

In [ ]:
# summary table for numeric columns (selecting only numeric columns before)
numeric_df = df_copy.select_dtypes(include=[np.number]) # select numeric columns
summary_stats = numeric_df.describe().T[['mean', '50%', 'std']].rename(columns={'50%': 'median'}) # summary table

# quantifying the relation of strength and fratility
correlation = df_copy['Grip strength'].corr(df_copy['Frailty_binary'])

print(summary_stats)
print(f"Correlation of strength and fratility: {correlation:.4f}")

                     mean     median       std
Grip strength   26.000000  27.000000  4.521553
Height_m         1.742440   1.738630  0.042435
Weight_kg       59.828834  61.688562  6.455441
BMI             19.682000  19.185000  1.780972
Frailty_binary   0.400000   0.000000  0.516398
Correlation of strength and fratility: -0.4759


In [ ]:
# create report
with open('findings.md', 'w') as f:
    f.write(f"""# Frailty Analysis Report

## summary table
{summary_stats.to_markdown()}

## strength and fratility
Correlation of strength and fratility: **{correlation:.4f}**
""")